# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs and schema.

To list record sets and their associated fields (referenced by their `@id`), we inspect the Croissant schema. The `mlcroissant` API exposes record sets and allows printing their schemas.

In [ ]:
# List available record sets and their fields by `@id`

print("Available record sets and fields (@id):\n")

for record_set in dataset.record_sets:
    print(f"Record Set: {record_set['@id']}")
    if 'field' in record_set:
        fields = record_set['field']
        # field could be a list or dict
        if isinstance(fields, list):
            for field in fields:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
                print(f" • Field: {field_id}")
        elif isinstance(fields, dict):
            print(f" • Field: {fields['@id']}")
    print()

# As an example, let's print some records for the first record set

if len(dataset.record_sets) > 0:
    record_set_id = dataset.record_sets[0]['@id']
    print(f"Sample records for record set: {record_set_id}\n")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for further analysis. All record sets and fields are referenced by their `@id`.

In [ ]:
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Only load if records exist
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}")

# For demonstration, select the first record set with records for further analysis
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes:
        main_record_set_id = rid
        break
if main_record_set_id:
    print(f"Columns in {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set with records found for further analysis.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping. This demonstration uses a numeric field, referenced by its `@id` (column name), from the main record set DataFrame.

In [ ]:
# Automatically select a numeric field from the main record set (by @id)
df = dataframes[main_record_set_id]
numeric_field = None
for col in df.columns:
    # Try to infer possible numeric columns
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
    # Try coercing if all values look like numbers
    try:
        df[col] = pd.to_numeric(df[col])
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    except Exception:
        continue

if numeric_field is None:
    print("No numeric field found for analysis.")
else:
    print(f"Using numeric field for analysis: {numeric_field}")
    # Example: Filter records where numeric field > threshold (e.g., 10)
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean())/
        filtered_df[numeric_field].std()
    )
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another non-numeric field if any
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        # Show group-wise mean
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by {group_field}, mean of '{numeric_field}':")
        display(grouped_df.head())
    else:
        print("No group field found for grouping analysis.")

## 5. Visualization

Visualize the distribution of the numeric field and explore relationships between fields, using only `@id` (column names) from the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field], bins=20, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion

In this notebook, you explored the FAIR<sup>2</sup> dataset on extracellular vesicle mass spectrometry from human mast cells using the `mlcroissant` library. You loaded metadata, listed record sets and fields (referenced by their `@id`), extracted data, and conducted a basic EDA: filtering by numeric criteria, normalization, grouping, and visualization. This workflow can be adapted to other Croissant-compliant datasets by substituting the schema URL and following the same pattern.

**Key takeaways:**
- The `mlcroissant` library makes schema-driven FAIR data easy to access in Python.
- Operations should reference all schema entities by their `@id`, ensuring unambiguous querying and portable code.
- The Croissant model supports examination of multiple record sets and fields, and is easily extensible for deeper analysis or machine learning pipelines.

For further analysis, consult the dataset's full documentation and schema at the [source URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).